# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

The paper reports that growing content was, on average, longer and younger than declining content. Growing pages averaged about 3.2K words and 184 days of age, compared with about 2.3K words and 230 days for declining pages.

**My methodology question:** How was the growing/declining label defined, and does the comparison control for differences between clients or other portfolio characteristics? I would treat this as an observed and measured directional pattern, not proof that increasing word count or reducing content age directly causes growth.

### Finding 2 — The Content Performance Curve

The paper reports that content reaches its strongest health score around 61–90 days and shows a decline around 271–365 days. It also reports a rebound for 365+ day content, while cautioning that older pages can recover when they are refreshed.

**My methodology question:** Does the validation design separate the effect of content age from the effect of refreshing content? Could the older-page rebound be influenced by which pages remained in the dataset or which pages were selected for refresh? I would treat this as an observed portfolio pattern and decision-support signal, not proof that content age itself causes decline or that refreshing always causes recovery.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    r"D:\Internship\Week1\Task1-Week1\data\raw\content_refresh_anonymized.csv"
)

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [7]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

# Features
X = df.drop(columns=[
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "is_declining_label"
])

# Convert categorical columns into dummy variables
X = pd.get_dummies(X, drop_first=True)

# Target
y = df["is_declining_label"]

# Client groups
groups = df["client_id"]





In [8]:
# Grouped train/test split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

# Train model
rf_grouped = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_grouped.fit(X_train, y_train)

# Predict
y_pred_grouped = rf_grouped.predict(X_test)



In [9]:
# Accuracy
grouped_accuracy = accuracy_score(y_test, y_pred_grouped)

print("Client-Grouped Split Accuracy:", grouped_accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_grouped))

Client-Grouped Split Accuracy: 0.5724484828817135

Classification Report:
              precision    recall  f1-score   support

           0       0.57      0.54      0.55      3014
           1       0.58      0.60      0.59      3149

    accuracy                           0.57      6163
   macro avg       0.57      0.57      0.57      6163
weighted avg       0.57      0.57      0.57      6163



### Before vs After

| Evaluation | Accuracy |
|---|---:|
| Week-5 random split | 70.13% |
| Client-grouped split | 57.24% |

The measured accuracy decreased from 70.13% under the original random split to 57.24% under the client-grouped split. The grouped split provides a more conservative evaluation because records from the same client are kept within either the training or testing group.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:
# Leakage Audit

leakage_candidates = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]



In [11]:
print("Potential leakage fields:")
for feature in leakage_candidates:
    print(feature)

Potential leakage fields:
trend_direction
trend_pct
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d


In [12]:
print("\nFeatures used by the final model:")
print(X.columns.tolist())


Features used by the final model:
['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'age_tier_order', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level_LOW', 'competition_level_MEDIUM', 'content_type_feedly article', 'content_type_keyword article', 'main_intent_informational', 'main_intent_navigational', 'main_intent_transactional', 'provider_used_openai', 'model_used_gemini-3-flash-preview', 'model_used_gpt-4o-mini', 'model_used_gpt-5-mini', 'model_used_unknown', 'age_tier_31-90', 'age_tier_365+', 'age_tier_91-180', 'freshness_tier_181+', 'freshness_tier_31-90', 'freshness_tier_91-180', 'word_count_tier_2000-3500', 'word_count_tier_3500+', 'word_count_tier_<1000', 'char_count_tier_25000+', '

### Leakage Audit Findings

The target label `is_declining_label` is created from `trend_direction`, so `trend_direction` and `trend_pct` were excluded from the final feature set.

The recent and previous 30-day performance fields were also excluded because they are closely related to the trend information used to define the target and could provide information about the outcome.

`client_id` and `content_id` were excluded because they are identifiers and were not used as predictive features.

The final feature set therefore excludes the identified target-derived and trend-related fields. This reduces the risk of leakage in the evaluated model.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*


### Original claim

The Random Forest model learned meaningful patterns from the historical data and generalized reasonably well on unseen test data.

### Revised claim

The Random Forest model showed a measured accuracy of 70.13% under the original random split, while accuracy decreased to 57.24% under a client-grouped split. The observed difference suggests that the original random split may have provided an optimistic estimate of performance. Under the client-grouped evaluation, the model provides directional decision-support potential, but the evidence is not sufficient to claim strong generalization to unseen clients.

## Self-check

Before you submit, confirm each line honestly:

[x] Every section above is filled
[x] The notebook runs top to bottom with no errors
[x] No client names, URLs, or private queries
[x] Claims use careful words
[X] Committed to my repo